# IFEval — instruction following (local)

IFEval gives the model 541 prompts that contain verifiable instructions ("write at least 300 words", "use no commas", "end with the phrase ...") and checks each response with code.

This notebook runs the benchmark **locally**, straight against the model's chat
endpoint — no EvalHub and no OpenShift. It is self-contained: all settings are in the
configuration cell below, and no job file is needed. It is the notebook equivalent of
the matching `run-*-local.sh` script.

**How it works:** [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)
(`lm_eval`) is run through `uvx`, which fetches it on demand into a throwaway
environment. Questions are sent to the model's `/chat/completions` endpoint with the
model's chat template applied, and answers are scored locally.

**Requires:** [`uv`](https://docs.astral.sh/uv/) (for `uvx`) and network access to the
model endpoint.

## 1. Configuration

Everything you might want to change lives in this one cell.

In [ ]:
import json, os, shutil, subprocess
from getpass import getpass
from pathlib import Path

# --- Model endpoint ---
MODEL_URL = "https://maas.apps.ocp.home.glroland.com/sandbox/gemma-4/v1"  # OpenAI-compatible base URL
MODEL_NAME = "gemma-4"

# --- Benchmark ---
TASK = "ifeval"  # lm-eval task name
LM_EVAL_VERSION = "0.4.13"  # pinned so results are repeatable
LM_EVAL_EXTRAS = "api,ifeval"  # pip extras lm-eval needs for this task
NUM_CONCURRENT = 8          # parallel requests to the model
THRESHOLD = 0.6  # minimum score to count as PASS
LIMIT = None                # e.g. 10 for a quick smoke test; None runs the full benchmark
# API key for the model endpoint. Uses the MODEL_API_KEY environment variable if set,
# otherwise prompts for it (leave blank if the endpoint needs no key).
MODEL_API_KEY = os.environ.get("MODEL_API_KEY") or getpass(f"Model API key for {MODEL_NAME} (leave blank if none): ")

## Helper

`run()` launches a command and streams its output into the notebook as it arrives, so you
can watch progress. It raises an error if the command exits non-zero.

In [ ]:
def run(cmd, env=None):
    proc = subprocess.Popen(cmd, env={**os.environ, **(env or {})}, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{cmd[0]} exited with {proc.returncode}")

## 2. Run the benchmark

Builds the `lm_eval` command and runs it. Completed requests are cached on disk, so if
the run dies partway (for example the gateway drops a connection), re-running this cell
resumes where it left off. The cache is deleted after a successful run so the next run
always gets fresh answers from the model.

In [ ]:
OUTPUT = Path(f".eval-notebook-{TASK}")             # lm-eval writes results and samples here
CACHE = Path(f".eval-notebook-{TASK}-cache") / "requests"
CACHE.parent.mkdir(exist_ok=True)
shutil.rmtree(OUTPUT, ignore_errors=True)

cmd = ["uvx", "--python", "3.12", "--from", f"lm-eval[{LM_EVAL_EXTRAS}]=={LM_EVAL_VERSION}",
       "lm_eval", "--model", "local-chat-completions",
       "--model_args", f"model={MODEL_NAME},base_url={MODEL_URL.rstrip('/')}/chat/completions,"
                       f"num_concurrent={NUM_CONCURRENT},max_retries=5,tokenized_requests=False",
       "--tasks", TASK, "--apply_chat_template", "--log_samples",
       "--output_path", str(OUTPUT), "--use_cache", str(CACHE)]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]

run(cmd, env={"OPENAI_API_KEY": MODEL_API_KEY or "not-required"})
shutil.rmtree(CACHE.parent, ignore_errors=True)

## 3. Score and pass/fail

The headline score is `inst_level_strict_acc` — the fraction of individual instructions followed exactly. The loose and prompt-level variants are printed too. Per-question model answers are in the `.eval-notebook-<task>` folder (`samples_*.jsonl`) if you want to see what it got wrong.

In [ ]:
results_file = next(OUTPUT.rglob("results_*.json"))
scores = json.loads(results_file.read_text())["results"][TASK]
print(json.dumps({k.split(",")[0]: v for k, v in scores.items()
                  if k.endswith(",none") and not k.startswith("alias")}, indent=2))

score = scores["inst_level_strict_acc,none"]
print("PASS" if score >= THRESHOLD else "FAIL", f"- inst_level_strict_acc {score:.3f} vs threshold {THRESHOLD}")